# Relações entre variáveis

Nesta etapa, passamos da descrição de **uma variável por vez** para a investigação de
**relações entre variáveis**. A pergunta deixa de ser apenas “como esta característica
se distribui?” e passa a incluir questões como:

- documentos com mais palavras tendem a ocupar mais páginas?
- documentos maiores tendem a mencionar mais pessoas?
- o tamanho dos documentos difere entre gêneros?
- a distribuição dos temas muda entre gêneros?

Usaremos um novo corpus sintético, criado especificamente para esta e para a próxima
etapa. O arquivo `dados/documentos.csv`, utilizado nas etapas anteriores, **não é
alterado**.

O objetivo aqui é aprender a **identificar, visualizar, quantificar e descrever**
relações. Explicações causais exigem evidências adicionais e serão tratadas com mais
cuidado na próxima etapa.


In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = "https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git"
REPOSITORIO = Path("/content/disciplina_computacao_aplicada_humanidades_digitais")
PASTA_UNIDADE = REPOSITORIO / "unidade_04"

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             URL_REPOSITORIO, str(REPOSITORIO)],
            check=True,
        )
    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")


## Geração do corpus desta etapa

Assim como no primeiro notebook da unidade, começamos tornando explícita a origem dos
dados. O corpus abaixo é **sintético**: não representa uma coleção histórica real.
Ele foi construído para permitir exercícios de exploração quantitativa mantendo
características plausíveis de documentos.

A geração é **reprodutível**: a mesma semente (`SEED`) produz sempre o mesmo conjunto
de 120 documentos. Algumas variáveis são geradas em conjunto, em vez de
independentemente. Isso é importante: se todas as colunas fossem sorteadas sem relação
entre si, haveria pouco a investigar.

A célula grava o resultado em `dados/documentos_relacoes.csv`. Ela não lê nem modifica
`dados/documentos.csv`.


In [ ]:
import math
import numpy as np
import pandas as pd

SEED = 20261058
N = 120
rng = np.random.default_rng(SEED)

# Seis documentos para cada ano entre 1890 e 1909.
linhas = []
for ano in range(1890, 1910):
    t = (ano - 1890) / 19
    p_capital = 0.12 + 0.76 * t

    # Há documentos dos dois locais em todos os anos.
    n_capital = int(np.clip(rng.binomial(6, p_capital), 1, 5))
    locais = np.array(
        ["Capital"] * n_capital + ["Interior"] * (6 - n_capital)
    )
    rng.shuffle(locais)
    linhas.extend((ano, local) for local in locais)

rng.shuffle(linhas)
anos = np.array([linha[0] for linha in linhas])
locais = np.array([linha[1] for linha in linhas])
capital = locais == "Capital"

# Gênero documental.
generos = rng.choice(
    ["notícia", "editorial", "carta"],
    size=N,
    p=[0.45, 0.30, 0.25],
)

# O tema não é sorteado com as mesmas probabilidades em todos os gêneros.
temas_possiveis = np.array(["educação", "trabalho", "progresso", "saúde"])
probabilidades_tema = {
    "notícia":   [0.20, 0.25, 0.15, 0.40],
    "editorial": [0.20, 0.30, 0.35, 0.15],
    "carta":     [0.40, 0.25, 0.20, 0.15],
}
temas = np.array([
    rng.choice(temas_possiveis, p=probabilidades_tema[genero])
    for genero in generos
])

# Tamanho do documento. Gênero, local, período e variação individual contribuem.
efeito_genero = {"notícia": 0.03, "editorial": 0.24, "carta": -0.18}
log_palavras = (
    math.log(650)
    + np.array([efeito_genero[g] for g in generos])
    + np.where(capital, 0.64, 0.0)
    - 0.016 * (anos - 1890)
    + rng.normal(0, 0.24, N)
)
palavras = np.maximum(140, np.rint(np.exp(log_palavras))).astype(int)

# Poucos documentos excepcionalmente longos.
especiais = rng.choice(N, 3, replace=False)
palavras[especiais] = np.rint(
    palavras[especiais] * rng.uniform(1.65, 2.05, 3)
).astype(int)

# A quantidade de palavras que cabe em uma página varia entre documentos.
densidade_pagina = np.clip(rng.normal(365, 38, N), 270, 470)
paginas = np.maximum(1, np.rint(palavras / densidade_pagina)).astype(int)

# Menções a pessoas são contagens e dependem, entre outros fatores, da extensão.
taxa_pessoas = {"notícia": 7.0, "editorial": 4.7, "carta": 5.8}
media_pessoas = palavras / 1000 * np.array(
    [taxa_pessoas[g] for g in generos]
)
pessoas = rng.poisson(media_pessoas)

dados = pd.DataFrame({
    "id_documento": [f"R{i:03d}" for i in range(1, N + 1)],
    "ano": anos,
    "genero": generos,
    "local": locais,
    "tema": temas,
    "palavras": palavras,
    "paginas": paginas,
    "pessoas": pessoas,
})

Path("dados").mkdir(exist_ok=True)
dados.to_csv("dados/documentos_relacoes.csv", index=False)

print("Arquivo gerado: dados/documentos_relacoes.csv")
print("Dimensões:", dados.shape)
dados.head()


## Antes de calcular: que tipo de relação estamos procurando?

O tipo das variáveis orienta a comparação.

| Variáveis comparadas | Exemplo | Primeira abordagem |
|---|---|---|
| quantitativa × quantitativa | `palavras` × `paginas` | gráfico de dispersão e correlação |
| categórica × quantitativa | `genero` × `palavras` | resumos por grupo e boxplot |
| categórica × categórica | `genero` × `tema` | tabela de contingência e proporções |

Não existe uma única medida chamada “a relação entre duas variáveis”. A ferramenta
adequada depende do significado e da escala das variáveis.

Começaremos pelo caso quantitativo × quantitativo.


## Duas variáveis quantitativas: primeiro, olhar

Considere a pergunta:

> **Documentos com mais palavras tendem a ocupar mais páginas?**

Antes de calcular um coeficiente, podemos observar os pares de valores em um
**gráfico de dispersão**. Cada ponto representa um documento: sua posição horizontal
indica o número de palavras e sua posição vertical, o número de páginas.

Procure três características:

1. **direção** — os valores tendem a crescer juntos, um cresce enquanto o outro
   diminui, ou não há direção evidente?
2. **forma** — a relação parece aproximadamente linear ou apresenta curvatura?
3. **dispersão e casos extremos** — os pontos ficam próximos de um padrão ou há
   grande variação?


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(dados["palavras"], dados["paginas"], alpha=0.75)
plt.xlabel("Número de palavras")
plt.ylabel("Número de páginas")
plt.title("Tamanho do documento: palavras × páginas")
plt.grid(alpha=0.2)
plt.show()


O gráfico sugere uma associação positiva: documentos com mais palavras tendem a ter
mais páginas. Podemos agora quantificar a intensidade da **associação linear**.

## Correlação de Pearson

O coeficiente de correlação de Pearson entre duas variáveis quantitativas $X$ e $Y$ é

$$
r =
\frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}
{\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2}
 \sqrt{\sum_{i=1}^{n}(y_i-\bar{y})^2}}.
$$

Seu valor está entre $-1$ e $1$:

- $r>0$: associação linear positiva;
- $r<0$: associação linear negativa;
- $r$ próximo de zero: pouca associação **linear**;
- quanto mais $|r|$ se aproxima de 1, mais os pontos se aproximam de uma reta.

Não adotaremos limites rígidos como “0,7 significa correlação forte”. A importância de
um valor depende do fenômeno estudado, da qualidade dos dados e da pergunta de
pesquisa. O coeficiente também não substitui o gráfico.


In [ ]:
r_palavras_paginas = dados["palavras"].corr(dados["paginas"])
print(f"Correlação entre palavras e páginas: {r_palavras_paginas:.3f}")


O número calculado resume a intensidade da relação linear observada no gráfico. Uma
forma cuidadosa de registrar o resultado é:

> **Neste corpus**, documentos com mais palavras tendem a ter mais páginas; a
> associação linear entre as duas medidas é positiva e elevada.

Observe o que a frase **não** diz: o coeficiente, sozinho, não demonstra que uma
variável causa a outra nem explica por que a associação existe.


## Comparando diferentes pares

Agora compare outras variáveis quantitativas. Antes de executar a próxima célula,
tente antecipar o resultado:

- `palavras` × `paginas`;
- `palavras` × `pessoas`;
- `paginas` × `pessoas`;
- `ano` × `palavras`.

Qual par você espera que apresente a associação linear mais evidente? Em qual deles a
interpretação exige maior cautela?


In [ ]:
pares = [
    ("palavras", "paginas"),
    ("palavras", "pessoas"),
    ("paginas", "pessoas"),
    ("ano", "palavras"),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

for ax, (x, y) in zip(axes.ravel(), pares):
    ax.scatter(dados[x], dados[y], alpha=0.70)
    r = dados[x].corr(dados[y])
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{x} × {y}   (r = {r:.3f})")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


Os quatro gráficos mostram por que convém combinar **visualização** e **medida**.
O coeficiente permite comparar associações lineares em uma escala comum, enquanto o
gráfico mostra forma, dispersão e observações individuais.

Podemos reunir os coeficientes das variáveis quantitativas em uma **matriz de
correlação**.


In [ ]:
variaveis_quantitativas = ["ano", "palavras", "paginas", "pessoas"]
matriz_correlacao = dados[variaveis_quantitativas].corr().round(3)
matriz_correlacao


A diagonal vale 1 porque cada variável tem correlação perfeita consigo mesma. A matriz
é simétrica: a correlação entre `palavras` e `paginas` é a mesma que entre `paginas`
e `palavras`.

A matriz é um resumo útil, mas não deve ser lida como uma coleção de respostas
automáticas. Cada célula precisa voltar à pergunta substantiva e, quando relevante,
ao gráfico correspondente.

## Valores extremos podem influenciar a correlação

Na etapa anterior vimos que um valor extremo não é automaticamente um erro. Agora
podemos observar outro motivo para inspecioná-lo: alguns casos podem exercer influência
considerável sobre medidas de associação.

Usaremos a mesma regra de $1{,}5\,IQR$ para localizar candidatos a valores extremos
em `palavras` e compararemos a correlação `palavras` × `paginas` antes e depois de
retirá-los **apenas como experimento analítico**.


In [ ]:
q1 = dados["palavras"].quantile(0.25)
q3 = dados["palavras"].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

extremos = dados[
    (dados["palavras"] < limite_inferior)
    | (dados["palavras"] > limite_superior)
]

display(extremos[[
    "id_documento", "ano", "genero", "local",
    "palavras", "paginas", "pessoas"
]])

r_com_todos = dados["palavras"].corr(dados["paginas"])
sem_extremos = dados.drop(index=extremos.index)
r_sem_extremos = sem_extremos["palavras"].corr(sem_extremos["paginas"])

print(f"Com todos os documentos: {r_com_todos:.3f}")
print(f"Sem candidatos a extremos: {r_sem_extremos:.3f}")


A comparação não autoriza apagar o documento extremo. Ela responde a uma pergunta
diferente:

> **Quanto nossa descrição da relação depende desse caso?**

Se o coeficiente mudar pouco, a associação é relativamente estável a essa exclusão.
Se mudar muito, devemos investigar o documento e relatar a sensibilidade do resultado.

## Correlação próxima de zero não significa necessariamente ausência de relação

Pearson mede **associação linear**. Para tornar essa limitação visível, construiremos
um exemplo auxiliar, separado do corpus.

Em `y = x²`, conhecer $x$ informa exatamente $y$. Entretanto, valores negativos e
positivos de $x$ produzem uma curva em U. As tendências lineares das duas metades se
compensam.


In [ ]:
x = np.linspace(-5, 5, 101)
y = x ** 2

print(f"Correlação de Pearson: {pd.Series(x).corr(pd.Series(y)):.3f}")

plt.figure(figsize=(7, 5))
plt.scatter(x, y)
plt.xlabel("x")
plt.ylabel("y = x²")
plt.title("Uma relação forte que não é linear")
plt.grid(alpha=0.2)
plt.show()


Portanto:

> $r\approx0$ não significa “as variáveis não têm relação”; significa que não há uma
> associação **linear** importante capturada por Pearson.

Esse é outro motivo para olhar os dados antes de resumir sua relação por um número.

## Uma variável categórica e uma quantitativa

Nem toda comparação quantitativa envolve duas medidas numéricas. Podemos perguntar:

> **O tamanho dos documentos difere entre gêneros?**

Como `genero` é categórica e `palavras` é quantitativa, calculamos resumos da
distribuição de palavras **dentro de cada grupo**.


In [ ]:
resumo_por_genero = (
    dados.groupby("genero")["palavras"]
    .agg(["count", "mean", "median", "std"])
    .rename(columns={
        "count": "n",
        "mean": "media",
        "median": "mediana",
        "std": "desvio_padrao",
    })
    .round(1)
)
resumo_por_genero


In [ ]:
ordem = ["carta", "notícia", "editorial"]

plt.figure(figsize=(8, 5))
plt.boxplot(
    [dados.loc[dados["genero"] == g, "palavras"] for g in ordem],
    tick_labels=ordem,
)
plt.xlabel("Gênero")
plt.ylabel("Número de palavras")
plt.title("Distribuição do tamanho dos documentos por gênero")
plt.grid(axis="y", alpha=0.2)
plt.show()


O boxplot permite comparar centro, dispersão e sobreposição entre os grupos. Note que
diferenças entre médias não implicam separação completa: documentos de gêneros
diferentes podem ter tamanhos semelhantes.

Também vale comparar média e mediana. Se elas conduzirem a impressões diferentes,
retorne à distribuição antes de escolher qual medida melhor responde à pergunta.

## Duas variáveis categóricas

Para `genero` e `tema`, retomamos a tabela de contingência. Agora ela é usada
explicitamente para investigar uma relação:

> **A composição temática é semelhante nos diferentes gêneros documentais?**


In [ ]:
contagens = pd.crosstab(dados["genero"], dados["tema"])
proporcoes = pd.crosstab(
    dados["genero"],
    dados["tema"],
    normalize="index",
).round(3)

print("Contagens:")
display(contagens)

print("Proporções dentro de cada gênero:")
display(proporcoes)


As contagens respondem **quantos** documentos aparecem em cada combinação. As
proporções por linha respondem **como os temas se distribuem dentro de cada gênero**.

Aqui não usamos Pearson: `genero` e `tema` são categorias nominais. Transformar seus
rótulos arbitrariamente em números para calcular uma correlação produziria um
coeficiente sem interpretação adequada.

Nesta etapa basta identificar e descrever diferenças na tabela. Uma medida específica
de intensidade de associação entre categorias não é necessária para responder à
pergunta introdutória.

## Um roteiro para investigar relações

Ao encontrar duas variáveis de interesse:

1. identifique o tipo e o significado de cada variável;
2. formule uma pergunta antes de calcular;
3. escolha uma representação compatível com os tipos das variáveis;
4. visualize a relação quando isso for informativo;
5. calcule um resumo adequado;
6. descreva o que os dados mostram sem transformar associação em explicação;
7. registre casos extremos, sobreposição e outras características que o resumo oculta.

A próxima etapa mostrará por que mesmo uma associação corretamente calculada pode
mudar de interpretação quando consideramos **denominadores, composição do corpus e
outras variáveis**.


## U04 — Atividade — relações entre variáveis

Escolha **três relações** no corpus, contemplando:

- uma relação quantitativa × quantitativa;
- uma relação categórica × quantitativa;
- uma relação categórica × categórica.

Para cada uma:

1. formule a pergunta de pesquisa;
2. identifique os tipos das duas variáveis;
3. escolha uma visualização ou tabela adequada;
4. calcule um resumo quando ele for apropriado;
5. escreva uma ou duas frases descrevendo o resultado;
6. separe explicitamente **descrição observada** de uma possível **hipótese
   explicativa**.

Para a relação quantitativa × quantitativa, inspecione o gráfico antes de interpretar
a correlação. Se escolher um par diferente dos exemplos do notebook, explique por que
Pearson é ou não é adequado.

**Pergunta para levar à próxima etapa:** que outra variável poderia alterar a
interpretação de uma das relações que você encontrou?
